## Setup

In [1]:
# autoreload so we don't have to restart the kernel when we change code in rag_helper.py
%load_ext autoreload
%autoreload 2

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

## Q1. How many lesson pages
How many lesson pages are in the dataset?

- 24
- 72 ✅
- 240
- 720

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']  
)

index.fit(documents)


In [6]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
print(index.search(question)[0]['content'][:30])
print(index.search(question)[0]['filename'])

# The Agentic Loop

Video: [Wa
01-agentic-rag/lessons/14-agentic-loop.md


## Q2. Indexing and searching

Index the documents with minsearch - make content a text field and
filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/14-agentic-loop.md ✅
- 04-evaluation/lessons/13-llm-as-judge.md
- 06-best-practices/lessons/02-hybrid-search.md

In [8]:
from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [9]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

# extra funcionality to get the last usage of the model
response = assistant.rag(question, return_usage=True)

In [10]:
# dedicated method to get the last usage of the model
assistant.get_last_usage()

ResponseUsage(input_tokens=623, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=716)

In [11]:
assistant.get_last_token_counts()

{'input_tokens': 623, 'output_tokens': 93, 'total_tokens': 716}

## Q3. How many input (prompt) tokens did we send to the model for
this request?

* 700 ✅ (approx 623)
* 7000
* 70000
* 700000

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

## Q4: How many chunks do you get?

- 70
- 295 ✅
- 1100
- 4500

In [15]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

chunk_index.fit(chunks)

In [16]:
chunk_index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [17]:
print(chunk_index.search(question)[0]['content'][:30])
print(chunk_index.search(question)[0]['filename'])

while` loop. The loop keeps ca
01-agentic-rag/lessons/14-agentic-loop.md


In [18]:
assistant_chunck = RAGBase(
    index=chunk_index,
    llm_client=openai_client
)

# extra funcionality to get the last usage of the model
response = assistant_chunck.rag(question, return_usage=True)

In [24]:
print(assistant_chunck.get_last_usage())
print("\n")
print(assistant_chunck.get_last_token_counts())

ResponseUsage(input_tokens=477, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=73, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=550)


{'input_tokens': 477, 'output_tokens': 73, 'total_tokens': 550}


In [28]:
assistant.get_last_usage().input_tokens / assistant_chunck.get_last_usage().input_tokens

1.3060796645702306

## Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

- about the same (+/-30% less from 623 -> 477) ✅
- 3× fewer
- 10× fewer
- 30× fewer

In [35]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [36]:
def search(query: str, k: int = 4) -> str:
    """
    Search the chunk index and return a short concatenated text of top-k snippets.
    """
    hits = chunk_index.search(query, num_results=k)
    snippets = []
    for h in hits:
        fname = h.get("filename", "<unknown>")
        excerpt = "\n".join((h.get("content") or "").splitlines()[:6])
        snippets.append(f"File: {fname}\n{excerpt}")
    return "\n\n---\n\n".join(snippets)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(prompt=question, callback=callback)

import pprint
pprint.pprint(getattr(result, "all_messages", None))

all_msgs = getattr(result, "all_messages", []) or []
count = 0
for m in all_msgs:
    if isinstance(m, dict):
        if m.get("type") == "tool_call" or m.get("role") == "tool" or m.get("tool_name"):
            count += 1
        meta = m.get("meta") or {}
        if isinstance(meta, dict) and meta.get("tool"):
            count += 1
    else:
        # attempt attribute access for objects
        try:
            typ = getattr(m, "type", None) or getattr(m, "role", None) or getattr(m, "tool_name", None)
            if typ:
                count += 1
        except Exception:
            pass

print("search_count =", count)
print("result.cost =", getattr(result, "cost", None))

-> Response received


-> Response received


[EasyInputMessage(content='\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n', role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop keep calling the model until it stops?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop keep calling the model until it stops model stops loop function calling repeatedly until done", "k": 5}', call_id='call_ZhxvxCDqGAx4U3tEdUEzYrTG', name='search', type='function_call', id='fc_0e566f500fb9c1e8006a3818e621f08192bb9036985f846515', namespace=None, status='completed'),
 {'call_id': 'call_ZhxvxCDqGAx4U3tEdUEzYrTG',
  'output': '"File: 01-agentic-rag/lessons/14-agentic-loop.md\\nwhile` loop. '
            'The loop keeps calling the model until\\nit returns a response '
   

## Q6: How many times did the agent call `search`?

* 0
* 4 ✅
* 10
* 20